# IEEE fraud model training: XGBoost hyperparameter tuning

Same train, validation, and test construction as `02_5_model_training_balancing_tests.ipynb`. The training set is cut once to `MAX_TRAIN_ROWS` with a stratified sample, so the fraud rate stays about 3.5%. Validation and test stay at the natural fraud rate. There is no resampling and no `scale_pos_weight`.

Optuna searches XGBoost on the columns in `TOP_40_FEATURES`. Edit that table in `src/model_training/hyperparameter_tuning.py`, or replace `FEATURE_TABLE` in the feature cell below. Each row is `[rank, feature name, gain]`. Only the feature name is trained on.

The search maximizes validation PR-AUC. Early stopping still watches validation logloss and keeps the best validation round. The test set is scored once, after the search, on the best parameters. The F1 threshold is chosen on the natural validation set.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). The final model is logged to `ieee_fraud_hyperparameter_tuning`, separate from the balancing comparison.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import pandas as pd

from src.model_training.hyperparameter_tuning import (
    TOP_40_FEATURES,
    feature_names,
    tune_xgboost,
)
from src.model_training.training import ModelTrainingWorkflow
from src.run_time_configuration import BaseConfigParams

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
MAX_TRAIN_ROWS = 100_000
N_TRIALS = 30
EARLY_STOPPING_ROUNDS = 5
EARLY_STOPPING_MIN_DELTA = 0.001
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]


In [2]:
def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Processed dir: /home/martinezyamamotoarthur/modeling_fraud_system/data/processed
MLflow tracking URI: http://localhost:5000


In [3]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))


train (331633, 332) pos_rate 0.0352
validation (82909, 332) pos_rate 0.0352
test (175998, 332) pos_rate 0.0344


In [4]:
from sklearn.model_selection import train_test_split


def class_counts(df: pd.DataFrame) -> pd.Series:
    return df[TARGET_COLUMN].value_counts().sort_index()


def limit_training_rows(df: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.reset_index(drop=True)
    kept, _ = train_test_split(
        df,
        train_size=max_rows,
        random_state=seed,
        stratify=df[TARGET_COLUMN],
    )
    return kept.reset_index(drop=True)


df_train_capped = limit_training_rows(df_train, MAX_TRAIN_ROWS, SEED)

print("train loaded", df_train.shape)
print(class_counts(df_train))
print("train capped", df_train_capped.shape)
print(class_counts(df_train_capped))
print("validation", df_validation.shape)
print(class_counts(df_validation))
print("test", df_test.shape)
print(class_counts(df_test))

del df_train


train loaded (331633, 332)
is_fraud
0    319953
1     11680
Name: count, dtype: int64
train capped (100000, 332)
is_fraud
0    96478
1     3522
Name: count, dtype: int64
validation (82909, 332)
is_fraud
0    79989
1     2920
Name: count, dtype: int64
test (175998, 332)
is_fraud
0    169935
1      6063
Name: count, dtype: int64


## Features

Edit `FEATURE_TABLE` to change the columns. Leave it as `TOP_40_FEATURES` to use the table in `src/model_training/hyperparameter_tuning.py`. That default is the gain ranking from the no-balancing run. Add, remove, or reorder rows. Each row is `[rank, feature name, gain]`. Rank and gain are notes; training uses the name only.


In [5]:
# Edit this assignment to train on a different column list.
FEATURE_TABLE = TOP_40_FEATURES
SELECTED_FEATURES = feature_names(FEATURE_TABLE)

print(f"{len(SELECTED_FEATURES)} features")
for rank, name, gain in FEATURE_TABLE:
    print(f"{int(rank):>2}  {name}  {gain}")


40 features
 1  card1_target_rate  2898.5080566406
 2  card1_x_product_channel_target_rate  2619.8493652344
 3  C2  1788.8270263672
 4  V217  1463.5167236328
 5  V279  1325.2569580078
 6  C11  1288.9261474609
 7  V29  1281.1726074219
 8  V283  1202.521484375
 9  V95  1136.4382324219
10  V312  1072.3162841797
11  C6  985.2222290039
12  V282  894.4429321289
13  C12  891.3984375
14  V258  871.4403686523
15  C14  862.3701782227
16  card_D10_std  859.451965332
17  V313  833.7211914062
18  card_C9_mean  800.0765991211
19  V57  793.2509155273
20  flag_outsider15  753.5994262695
21  V69  727.9138793945
22  V90  709.7984008789
23  D3  705.2971801758
24  V53  701.7498168945
25  C1  693.0908813477
26  card_C13_nunique  680.6305541992
27  V265  663.9107666016
28  V314  645.0280761719
29  amount_cents  589.6865234375
30  D2  581.2379150391
31  V35  529.9074707031
32  card_V309_nunique  527.7366943359
33  V285  516.6747436523
34  C13  514.0857543945
35  V288  506.8506774902
36  purchaser_email_domai

## Fit the search data

No balancing. Training rows stay at the capped natural fraud rate. Validation and test are unchanged. Preprocessing keeps the selected columns and numeric values only, including native missing values.


In [6]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud_hyperparameter_tuning",
    mlflow_run_name="xgb_top40_optuna",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)
workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train_capped
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train_capped)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(
    numeric_only=True,
    selected_features=SELECTED_FEATURES,
)

print("train", workflow.x_train_preproc.shape, "pos_rate", round(float(workflow.y_train.mean()), 4))
print("validation", workflow.x_validation_preproc.shape, "pos_rate", round(float(workflow.y_validation.mean()), 4))
print("test", workflow.x_test_preproc.shape, "pos_rate", round(float(workflow.y_test.mean()), 4))
print("columns", list(workflow.x_train_preproc.columns))


2026-10-11 17:42:55,766 - src.model_training.training - INFO - Initialized the SolvencyNewCustomersModelTraining workflow.
2026-10-11 17:42:55,774 - src.model_training.training - INFO - Generating preprocessed data by copying original data.
2026-10-11 17:42:55,776 - src.model_training.training - INFO - Using selected features: 40 features specified
2026-10-11 17:42:55,777 - src.model_training.training - INFO - Selected features: ['card1_target_rate', 'card1_x_product_channel_target_rate', 'C2', 'V217', 'V279', 'C11', 'V29', 'V283', 'V95', 'V312', 'C6', 'V282', 'C12', 'V258', 'C14', 'card_D10_std', 'V313', 'card_C9_mean', 'V57', 'flag_outsider15', 'V69', 'V90', 'D3', 'V53', 'C1', 'card_C13_nunique', 'V265', 'V314', 'amount_cents', 'D2', 'V35', 'card_V309_nunique', 'V285', 'C13', 'V288', 'purchaser_email_domain_target_rate', 'V309', 'V221', 'V284', 'V131']
2026-10-11 17:42:55,862 - src.model_training.training - INFO - Filtering to numeric columns: 40 out of 40 total columns
2026-10-11 17

train (100000, 40) pos_rate 0.0352
validation (82909, 40) pos_rate 0.0352
test (175998, 40) pos_rate 0.0344
columns ['card1_target_rate', 'card1_x_product_channel_target_rate', 'C2', 'V217', 'V279', 'C11', 'V29', 'V283', 'V95', 'V312', 'C6', 'V282', 'C12', 'V258', 'C14', 'card_D10_std', 'V313', 'card_C9_mean', 'V57', 'flag_outsider15', 'V69', 'V90', 'D3', 'V53', 'C1', 'card_C13_nunique', 'V265', 'V314', 'amount_cents', 'D2', 'V35', 'card_V309_nunique', 'V285', 'C13', 'V288', 'purchaser_email_domain_target_rate', 'V309', 'V221', 'V284', 'V131']


## Search

Each trial trains XGBoost with early stopping on the validation set. Optuna keeps the trial with the highest validation PR-AUC. `N_TRIALS`, `EARLY_STOPPING_ROUNDS`, and `EARLY_STOPPING_MIN_DELTA` are set in the first cell.


In [7]:
study = tune_xgboost(
    workflow,
    n_trials=N_TRIALS,
    seed=SEED,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

best_hyperparameters = study.best_trial.user_attrs["hyperparameters"]
print("best validation PR-AUC", round(study.best_value, 4))
print("best iteration", study.best_trial.user_attrs["best_iteration"])
print(best_hyperparameters)

trials = study.trials_dataframe().sort_values("value", ascending=False)
trials.head(10)


2026-10-11 17:42:55,912 - src.model_training.hyperparameter_tuning - INFO - Starting XGBoost Optuna search: 30 trials, 82909 rows, 40 features.
[I 2026-10-11 17:42:55,914] A new study created in memory with name: xgboost_no_balancing


  0%|          | 0/30 [00:00<?, ?it/s]

/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:42:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
2026-10-11 17:42:56,919 - src.model_training.hyperparameter_tuning - INFO - Trial 0 validation PR-AUC 0.6075 best_iteration 31
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:42:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-10-11 17:42:56,920] Trial 0 finished with value: 0.6074754716401135 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'min_child_weight': 17.524101118128137, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.5780093202212182, 'gamma': 0.7799726016810132, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.6074754716401135.


2026-10-11 17:42:58,016 - src.model_training.hyperparameter_tuning - INFO - Trial 1 validation PR-AUC 0.6318 best_iteration 36
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:42:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


...

2026-10-11 17:43:28,843 - src.model_training.hyperparameter_tuning - INFO - Trial 28 validation PR-AUC 0.6374 best_iteration 31
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:43:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-10-11 17:43:28,844] Trial 28 finished with value: 0.6374257003695993 and parameters: {'max_depth': 7, 'learning_rate': 0.1130124438315864, 'min_child_weight': 2.5395046642628785, 'subsample': 0.8467794959112945, 'colsample_bytree': 0.699150557082143, 'gamma': 0.6270876505205121, 'reg_alpha': 0.001349247221816019, 'reg_lambda': 0.005441935781703866}. Best is trial 23 with value: 0.6467010550871781.


2026-10-11 17:43:29,979 - src.model_training.hyperparameter_tuning - INFO - Trial 29 validation PR-AUC 0.6520 best_iteration 37
2026-10-11 17:43:29,982 - src.model_training.hyperparameter_tuning - INFO - Optuna search finished. Best validation PR-AUC 0.6520 with {'max_depth': 8, 'learning_rate': 0.0787004377385462, 'min_child_weight': 2.334718704341846, 'subsample': 0.88610188253155, 'colsample_bytree': 0.7884436008182842, 'gamma': 0.5646241790185449, 'reg_alpha': 0.000271033976916434, 'reg_lambda': 0.0016433122938417782, 'n_estimators': 200, 'n_jobs': 4}.


[I 2026-10-11 17:43:29,980] Trial 29 finished with value: 0.6520394390910847 and parameters: {'max_depth': 8, 'learning_rate': 0.0787004377385462, 'min_child_weight': 2.334718704341846, 'subsample': 0.88610188253155, 'colsample_bytree': 0.7884436008182842, 'gamma': 0.5646241790185449, 'reg_alpha': 0.000271033976916434, 'reg_lambda': 0.0016433122938417782}. Best is trial 29 with value: 0.6520394390910847.
best validation PR-AUC 0.652
best iteration 37
{'max_depth': 8, 'learning_rate': 0.0787004377385462, 'min_child_weight': 2.334718704341846, 'subsample': 0.88610188253155, 'colsample_bytree': 0.7884436008182842, 'gamma': 0.5646241790185449, 'reg_alpha': 0.000271033976916434, 'reg_lambda': 0.0016433122938417782, 'n_estimators': 200, 'n_jobs': 4}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,user_attrs_best_iteration,user_attrs_hyperparameters,user_attrs_val_pr_auc,system_attrs_tpe:relative_params:0,state
29,29,0.652039,2026-10-11 17:43:28.846049,2026-10-11 17:43:29.980472,0 days 00:00:01.134423,0.788444,0.564624,0.078700,8,2.334719,0.000271,0.001643,0.886102,37,"{'max_depth': 8, 'learning_rate': 0.0787004377...",0.652039,"{""colsample_bytree"": 0.7884436008182842, ""gamm...",COMPLETE
23,23,0.646701,2026-10-11 17:43:23.484012,2026-10-11 17:43:24.470632,0 days 00:00:00.986620,0.709839,0.590430,0.120580,8,2.126383,0.108882,0.126430,0.848037,26,"{'max_depth': 8, 'learning_rate': 0.1205801456...",0.646701,"{""colsample_bytree"": 0.7098391056410238, ""gamm...",COMPLETE
25,25,0.645562,2026-10-11 17:43:25.228559,2026-10-11 17:43:26.190961,0 days 00:00:00.962402,0.775636,0.211054,0.072745,8,3.032022,0.000790,0.011456,0.865639,35,"{'max_depth': 8, 'learning_rate': 0.0727446691...",0.645562,"{""colsample_bytree"": 0.7756355530241937, ""gamm...",COMPLETE
27,27,0.645345,2026-10-11 17:43:27.094922,2026-10-11 17:43:28.004897,0 days 00:00:00.909975,0.720235,0.518508,0.097261,8,2.846384,0.010013,0.012248,0.858663,29,"{'max_depth': 8, 'learning_rate': 0.0972613442...",0.645345,"{""colsample_bytree"": 0.7202345732419524, ""gamm...",COMPLETE
24,24,0.643237,2026-10-11 17:43:24.473043,2026-10-11 17:43:25.226627,0 days 00:00:00.753584,0.783607,1.001065,0.129992,8,1.708392,0.017268,0.014066,0.824081,24,"{'max_depth': 8, 'learning_rate': 0.1299917493...",0.643237,"{""colsample_bytree"": 0.7836074687467626, ""gamm...",COMPLETE
26,26,0.642110,2026-10-11 17:43:26.192202,2026-10-11 17:43:27.093421,0 days 00:00:00.901219,0.829042,0.731423,0.151374,8,1.510376,0.829163,0.003584,0.845499,27,"{'max_depth': 8, 'learning_rate': 0.1513743604...",0.642110,"{""colsample_bytree"": 0.8290424108985274, ""gamm...",COMPLETE
19,19,0.641374,2026-10-11 17:43:19.972492,2026-10-11 17:43:20.957880,0 days 00:00:00.985388,0.753845,0.139699,0.100068,7,1.608785,0.002196,0.040027,0.876194,33,"{'max_depth': 7, 'learning_rate': 0.1000681367...",0.641374,"{""colsample_bytree"": 0.7538446359506572, ""gamm...",COMPLETE
13,13,0.641200,2026-10-11 17:43:13.632441,2026-10-11 17:43:14.636662,0 days 00:00:01.004221,0.745779,0.396996,0.112996,7,1.222235,0.021344,0.015634,0.883843,31,"{'max_depth': 7, 'learning_rate': 0.1129961949...",0.641200,"{""colsample_bytree"": 0.7457791489060237, ""gamm...",COMPLETE
28,28,0.637426,2026-10-11 17:43:28.006805,2026-10-11 17:43:28.844330,0 days 00:00:00.837525,0.699151,0.627088,0.113012,7,2.539505,0.001349,0.005442,0.846779,31,"{'max_depth': 7, 'learning_rate': 0.1130124438...",0.637426,"{""colsample_bytree"": 0.699150557082143, ""gamma...",COMPLETE
21,21,0.637109,2026-10-11 17:43:21.692500,2026-10-11 17:43:22.649263,0 days 00:00:00.956763,0.716906,0.944506,0.115566,7,1.718148,0.165647,0.077853,0.886371,30,"{'max_depth': 7, 'learning_rate': 0.1155658057...",0.637109,"{""colsample_bytree"": 0.7169063651790907, ""gamm...",COMPLETE


## Train the best parameters

One final fit, logged to MLflow. Test metrics are computed here and were not used to pick the trial. The kept model is the best validation round.


In [8]:
results = workflow.train_model(
    hyperparameters=best_hyperparameters,
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="xgb_top40_optuna",
    top_n_features=len(SELECTED_FEATURES),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)
test_metrics = results["test_metrics"]
summary = {
    "run_id": workflow.run_time_config.mlflow_run_id,
    "n_features": len(SELECTED_FEATURES),
    "train_rows": int(len(workflow.y_train)),
    "train_pos_rate": float(workflow.y_train.mean()),
    "best_iteration": results["best_iteration"],
    "threshold": float(results["threshold"]),
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
    "pr_auc": test_metrics["pr_auc"],
    "auc": test_metrics["auc"],
    "ks_statistic": test_metrics["ks_statistic"],
    "optuna_val_pr_auc": study.best_value,
}
summary


2026-10-11 17:43:30,058 - src.model_training.training - INFO - Starting model training without hyperparameter tuning using xgboost.
2026-10-11 17:43:30,088 - src.model_training.training - INFO - Training xgboost model with fixed hyperparameters and early stopping (patience 5, minimum validation improvement 0.001).
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:43:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
2026-10-11 17:43:31,208 - src.model_training.training - INFO - Best iteration on the validation set: 37.
2026-10-11 17:43:31,247 - src.model_training.training - INFO - Calculated optimal threshold (F1-based) on validation set: 0.2804
2026-10-11 17:43:31,456 - src.model_training.evaluation - INFO - Training Metrics - AUC: 0.9522, PR-AUC: 0.7395, KS: 0.7591
2026-10-11 17:43:31,535 - src.model_tr

🏃 View run xgb_top40_optuna at: http://localhost:5000/#/experiments/155/runs/b2b69069acb94c78bf89309d36886437
🧪 View experiment at: http://localhost:5000/#/experiments/155


{'run_id': 'b2b69069acb94c78bf89309d36886437',
 'n_features': 40,
 'train_rows': 100000,
 'train_pos_rate': 0.03522,
 'best_iteration': 37,
 'threshold': 0.2804490029811859,
 'precision': 0.6184280403611259,
 'recall': 0.384133267359393,
 'f1': 0.47390375419676467,
 'pr_auc': 0.46915376440290385,
 'auc': 0.8719609919056817,
 'ks_statistic': np.float32(0.5943156),
 'optuna_val_pr_auc': 0.6520394390910847}